In [1]:
!pip install --upgrade pip

  Using cached pip-26.2.2-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.2.2-py3-none-any.whl (1.8 MB)


ERROR: To modify pip, please run the following command:
C:\Git\ray-embedding-experiments\.venv-ray-test\Scripts\python.exe -m pip install --upgrade pip


In [2]:
!pip install "ray[data]"


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Dataset Generator

In [3]:
import json
import random
import subprocess
import os

# ─── CONFIGURAÇÃO ────────────────────────────────────────────────────────────

NUM_DOCS    = 5000        # Quantidade de documentos (aumente para experimentos maiores)
CHUNK_SIZE  = 150         # Palavras por chunk
OUTPUT_FILE = "dataset.jsonl"

# Workers para copiar o dataset (mesmo endereço/usuário do script de setup)
WORKER_NODES = [
    "192.168.1.11",
    "192.168.1.12",
    "192.168.1.13",
]
SSH_USER   = "ubuntu"
SSH_KEY    = os.path.expanduser("~/.ssh/id_rsa")
REMOTE_DIR = "~/ray-exp/"

# ─── TEMPLATES DE TEXTO ──────────────────────────────────────────────────────

TEMPLATES = [
    "O processamento distribuído permite escalar cargas de trabalho de inteligência artificial em múltiplos nós",
    "Ray Data organiza datasets em blocks armazenados no object store distribuído do cluster",
    "O modelo multilingual-e5-small gera vetores de 384 dimensões para representação semântica",
    "Embeddings são representações vetoriais densas que capturam significado semântico de textos",
    "O particionamento de dados melhora o throughput em pipelines de Retrieval-Augmented Generation",
    "Sistemas distribuídos exigem balanceamento de carga adequado entre workers para evitar gargalos",
    "A busca vetorial aproximada permite recuperação eficiente de documentos semanticamente similares",
    "O overhead de comunicação entre nós limita o ganho linear com aumento de paralelismo",
    "Apache Cassandra oferece armazenamento vetorial distribuído para pipelines de IA em produção",
    "O scheduler do Ray distribui tarefas considerando recursos disponíveis em cada nó do cluster",
    "map_batches() permite processar grandes datasets de forma paralela com controle de batch size",
    "O object store compartilhado do Ray permite transferência eficiente de dados entre workers",
    "Pipelines RAG combinam recuperação de informação com geração de linguagem para respostas precisas",
    "O autoscaler do Ray ajusta dinamicamente o número de nós conforme a demanda de processamento",
    "Vetores de alta dimensionalidade requerem índices especializados como HNSW para busca eficiente",
]

# ─── GERAÇÃO ─────────────────────────────────────────────────────────────────

def make_chunk(idx: int) -> dict:
    sentences = [random.choice(TEMPLATES) for _ in range(CHUNK_SIZE // 15 + 1)]
    text = ". ".join(sentences) + "."
    return {"id": idx, "text": text}


def main():
    print(f"Gerando {NUM_DOCS} documentos → {OUTPUT_FILE}")

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        for i in range(NUM_DOCS):
            f.write(json.dumps(make_chunk(i), ensure_ascii=False) + "\n")
            if (i + 1) % 1000 == 0:
                print(f"  {i + 1}/{NUM_DOCS} documentos gerados...")

    size_kb = os.path.getsize(OUTPUT_FILE) / 1024
    print(f"✓ Dataset gerado: {NUM_DOCS} docs, {size_kb:.1f} KB")

    # Distribuir para worker nodes via SCP
    if WORKER_NODES:
        print(f"\nDistribuindo dataset para {len(WORKER_NODES)} worker(s)...")
        for node in WORKER_NODES:
            cmd = [
                "scp", "-i", SSH_KEY,
                "-o", "StrictHostKeyChecking=no",
                OUTPUT_FILE,
                f"{SSH_USER}@{node}:{REMOTE_DIR}"
            ]
            result = subprocess.run(cmd, capture_output=True, text=True)
            if result.returncode == 0:
                print(f"  ✓ Copiado para {node}")
            else:
                print(f"  ✗ Erro em {node}: {result.stderr.strip()}")

    print("\nPróximo passo pipeline base")


if __name__ == "__main__":
    main()

Gerando 5000 documentos → dataset.jsonl
  1000/5000 documentos gerados...
  2000/5000 documentos gerados...
  3000/5000 documentos gerados...
  4000/5000 documentos gerados...
  5000/5000 documentos gerados...
✓ Dataset gerado: 5000 docs, 5337.7 KB

Distribuindo dataset para 3 worker(s)...
  ✗ Erro em 192.168.1.11: ssh: connect to host 192.168.1.11 port 22: Connection timed out
scp: Connection closed
  ✗ Erro em 192.168.1.12: ssh: connect to host 192.168.1.12 port 22: Connection timed out
scp: Connection closed
  ✗ Erro em 192.168.1.13: ssh: connect to host 192.168.1.13 port 22: Connection timed out
scp: Connection closed

Próximo passo pipeline base


Pipeline Base

In [ ]:
import ray
import time
import json
import psutil
import os

# ─── CONFIGURAÇÃO ────────────────────────────────────────────────────────────

HEAD_NODE_ADDRESS = "192.168.1.10:6379"   # IP:porta do head node
DATASET_FILE      = "dataset.jsonl"
BATCH_SIZE        = 64
CONCURRENCY       = 2                      # workers Ray iniciais
OUTPUT_FILE       = "baseline_result.json"

# ─── INICIALIZAÇÃO ────────────────────────────────────────────────────────────

print("=" * 50)
print("  PIPELINE BASE — VERIFICAÇÃO DO CLUSTER")
print("=" * 50)

ray.init(address=HEAD_NODE_ADDRESS, ignore_reinit_error=True)

# Inspeciona o cluster
nodes = ray.nodes()
alive = [n for n in nodes if n["Alive"]]
total_cpus = sum(n["Resources"].get("CPU", 0) for n in alive)

print(f"\nCluster conectado:")
print(f"  Nós ativos  : {len(alive)}")
print(f"  CPUs totais : {total_cpus:.0f}")
for n in alive:
    addr = n["NodeManagerAddress"]
    cpus = n["Resources"].get("CPU", 0)
    mem  = n["Resources"].get("memory", 0) / (1024 ** 3)
    role = "HEAD" if n.get("is_head_node") else "WORKER"
    print(f"  [{role}] {addr} → {cpus:.0f} CPUs, {mem:.1f} GB RAM")

# ─── MODELO DE EMBEDDING ─────────────────────────────────────────────────────

class EmbeddingModel:
    def __init__(self):
        from sentence_transformers import SentenceTransformer
        self.model = SentenceTransformer("intfloat/multilingual-e5-small")
        # Log qual nó está executando este worker
        import socket
        self.hostname = socket.gethostname()

    def __call__(self, batch: dict) -> dict:
        texts = batch["text"].tolist()
        embeddings = self.model.encode(
            texts,
            batch_size=BATCH_SIZE,
            show_progress_bar=False,
            normalize_embeddings=True,
        )
        batch["embedding"] = embeddings
        batch["worker_host"] = [self.hostname] * len(texts)
        return batch

# ─── EXECUÇÃO ────────────────────────────────────────────────────────────────

print(f"\nIniciando pipeline base...")
print(f"  Dataset   : {DATASET_FILE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Workers   : {CONCURRENCY}")

mem_before = psutil.virtual_memory().used / (1024 ** 2)
start = time.time()

ds = ray.data.read_json(DATASET_FILE)

ds_emb = ds.map_batches(
    EmbeddingModel,
    batch_size=BATCH_SIZE,
    concurrency=CONCURRENCY,
    batch_format="numpy",
)

result = ds_emb.take_all()
elapsed = time.time() - start
mem_after = psutil.virtual_memory().used / (1024 ** 2)

# ─── RESULTADOS ──────────────────────────────────────────────────────────────

total = len(result)
throughput = total / elapsed
mem_delta = mem_after - mem_before

# Conta quantos workers distintos processaram dados
workers_used = set()
for row in result:
    if "worker_host" in row:
        workers_used.add(row["worker_host"])

print("\n" + "=" * 50)
print("  RESULTADO BASELINE")
print("=" * 50)
print(f"  Total de embeddings : {total}")
print(f"  Tempo total (s)     : {elapsed:.2f}")
print(f"  Throughput (emb/s)  : {throughput:.1f}")
print(f"  Memória delta (MB)  : {mem_delta:.1f}")
print(f"  Workers utilizados  : {len(workers_used)}")
print(f"  Hosts: {workers_used}")

# Salva resultado baseline para referência de speedup
baseline = {
    "total_docs": total,
    "elapsed_s": elapsed,
    "throughput_emb_s": throughput,
    "mem_delta_mb": mem_delta,
    "workers_used": list(workers_used),
    "cluster_nodes": len(alive),
    "cluster_total_cpus": total_cpus,
    "batch_size": BATCH_SIZE,
    "concurrency": CONCURRENCY,
}

with open(OUTPUT_FILE, "w") as f:
    json.dump(baseline, f, indent=2)

print(f"\n✓ Baseline salvo em: {OUTPUT_FILE}")
print("  Próximo passo")

ray.shutdown()

c:\Git\ray-embedding-experiments\.venv-ray-test\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-10-28 14:46:09,047	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-10-28 14:46:09,085	INFO worker.py:1836 -- Connecting to existing Ray cluster at address: 192.168.1.10:6379...


  PIPELINE BASE — VERIFICAÇÃO DO CLUSTER


Experiment Paralelism

In [ ]:
import ray
import time
import json
import psutil
import statistics
import socket

# ─── CONFIGURAÇÃO ────────────────────────────────────────────────────────────

HEAD_NODE_ADDRESS = "192.168.1.10:6379"
DATASET_FILE      = "dataset.jsonl"

N_BLOCKS     = 8     # fixo
BATCH_SIZE   = 64    # fixo
N_RUNS       = 5
RESULTS_FILE = "results_exp3_paralelismo.jsonl"

# Workers serão definidos dinamicamente baseado no cluster real
# Você pode fixar manualmente: CONCURRENCY_VALUES = [1, 2, 4, 8]
CONCURRENCY_VALUES = None   # None = detecta automaticamente do cluster

# ─── MODELO ──────────────────────────────────────────────────────────────────

class EmbeddingModel:
    def __init__(self):
        from sentence_transformers import SentenceTransformer
        self.model = SentenceTransformer("intfloat/multilingual-e5-small")
        self.hostname = socket.gethostname()

    def __call__(self, batch: dict) -> dict:
        batch["embedding"] = self.model.encode(
            batch["text"].tolist(),
            batch_size=BATCH_SIZE,
            show_progress_bar=False,
            normalize_embeddings=True,
        )
        batch["worker_host"] = [self.hostname] * len(batch["text"])
        return batch

# ─── INICIALIZAÇÃO ────────────────────────────────────────────────────────────

ray.init(address=HEAD_NODE_ADDRESS, ignore_reinit_error=True)

nodes = ray.nodes()
alive = [n for n in nodes if n["Alive"]]
total_cpus = int(sum(n["Resources"].get("CPU", 0) for n in alive))

print("=" * 60)
print("  EXP 3 — PARALELISMO / WORKERS (QP3)")
print(f"  Cluster       : {len(alive)} nós, {total_cpus} CPUs totais")
print(f"  Blocos fixos  : {N_BLOCKS}")
print(f"  Batch fixo    : {BATCH_SIZE}")
print(f"  Repetições    : {N_RUNS}")

# Define os valores de concurrency automaticamente se não configurado
if CONCURRENCY_VALUES is None:
    # Cobre 1 worker, metade dos CPUs, todos os CPUs e, se possível, além
    candidates = sorted({1, max(1, total_cpus // 4),
                          max(1, total_cpus // 2), total_cpus})
    CONCURRENCY_VALUES = [c for c in candidates if c <= total_cpus]

print(f"  Workers testados: {CONCURRENCY_VALUES}")
print("=" * 60)

# ─── EXECUÇÃO ────────────────────────────────────────────────────────────────

baseline_time = None   # Tempo com 1 worker — denominador do speedup
all_results   = []

with open(RESULTS_FILE, "w") as out:
    for n_workers in CONCURRENCY_VALUES:
        times       = []
        throughputs = []
        mem_peaks   = []

        print(f"\n── Testando concurrency = {n_workers} ──")

        for run in range(N_RUNS):
            ds = ray.data.read_json(DATASET_FILE).repartition(N_BLOCKS)

            t0 = time.time()
            result = ds.map_batches(
                EmbeddingModel,
                batch_size=BATCH_SIZE,
                concurrency=n_workers,
                batch_format="numpy",
            ).take_all()
            elapsed = time.time() - t0

            # Conta hosts distintos que processaram dados
            hosts_used = {r.get("worker_host", "?") for r in result}

            times.append(elapsed)
            throughputs.append(len(result) / elapsed)
            mem_peaks.append(psutil.virtual_memory().percent)

            print(f"  run {run + 1}/{N_RUNS}: {elapsed:.2f}s | "
                  f"{len(result)/elapsed:.1f} emb/s | "
                  f"hosts: {hosts_used}")

        mean_t = statistics.mean(times)
        std_t  = statistics.stdev(times) if len(times) > 1 else 0.0

        # Baseline = execução com 1 worker
        if n_workers == 1 or baseline_time is None:
            if n_workers == 1:
                baseline_time = mean_t

        # Speedup: quanto mais rápido que o single-worker
        speedup_real  = (baseline_time / mean_t) if baseline_time else 1.0
        speedup_ideal = float(n_workers)          # crescimento linear esperado
        efficiency    = speedup_real / speedup_ideal if speedup_ideal else 0.0

        rec = {
            "experiment"            : "exp3_paralelismo",
            "n_workers"             : n_workers,
            "n_blocks"              : N_BLOCKS,
            "batch_size"            : BATCH_SIZE,
            "cluster_nodes"         : len(alive),
            "cluster_total_cpus"    : total_cpus,
            "mean_time_s"           : round(mean_t, 4),
            "std_time_s"            : round(std_t, 4),
            "mean_throughput_emb_s" : round(statistics.mean(throughputs), 2),
            "std_throughput_emb_s"  : round(statistics.stdev(throughputs) if len(throughputs) > 1 else 0, 2),
            "speedup_real"          : round(speedup_real, 4),
            "speedup_ideal"         : round(speedup_ideal, 4),
            "efficiency_pct"        : round(efficiency * 100, 2),   # 100% = linear perfeito
            "mean_ram_peak_pct"     : round(statistics.mean(mem_peaks), 2),
            "raw_times_s"           : [round(t, 4) for t in times],
        }

        out.write(json.dumps(rec) + "\n")
        out.flush()
        all_results.append(rec)

        print(f"  ► média: {mean_t:.2f}s ± {std_t:.2f} | "
              f"speedup: {speedup_real:.2f}x (ideal: {speedup_ideal:.0f}x) | "
              f"eficiência: {efficiency*100:.0f}%")

ray.shutdown()

# ─── RESUMO ──────────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("  RESUMO EXP 3")
print(f"  {'Workers':>8} {'Tempo (s)':>10} {'Std':>8} "
      f"{'emb/s':>8} {'Speedup':>10} {'Ideal':>8} {'Efic%':>8}")
print("-" * 70)
for r in all_results:
    print(f"  {r['n_workers']:>8} "
          f"{r['mean_time_s']:>10.2f} "
          f"{r['std_time_s']:>8.2f} "
          f"{r['mean_throughput_emb_s']:>8.1f} "
          f"{r['speedup_real']:>10.2f}x "
          f"{r['speedup_ideal']:>7.0f}x "
          f"{r['efficiency_pct']:>8.1f}%")

print(f"\n✓ Resultados salvos em: {RESULTS_FILE}")
print("  Próximo passo analise")

Experiment analysis

In [ ]:
import json
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt

# ─── ARQUIVO DE ENTRADA ───────────────────────────────────────────────────────

RESULTS_FILE = "results_exp3_paralelismo.jsonl"

if not os.path.exists(RESULTS_FILE):
    print(f"[ERRO] Arquivo não encontrado: {RESULTS_FILE}")
    print("  Execute primeiro: python3 5_exp3_paralelismo.py")
    sys.exit(1)

rows = [json.loads(line) for line in open(RESULTS_FILE, encoding="utf-8")]
df = pd.DataFrame(rows)
print(f"✓ {len(df)} configurações carregadas de {RESULTS_FILE}")
print(df[["n_workers", "mean_time_s", "speedup_real", "speedup_ideal", "efficiency_pct"]].to_string(index=False))

# ─── TEMA ────────────────────────────────────────────────────────────────────

plt.rcParams.update({
    "figure.facecolor" : "#1a1d27",
    "axes.facecolor"   : "#12141f",
    "axes.edgecolor"   : "#2d3150",
    "axes.labelcolor"  : "#c9d1e9",
    "xtick.color"      : "#8b91b8",
    "ytick.color"      : "#8b91b8",
    "text.color"       : "#e8eaf6",
    "grid.color"       : "#2d3150",
    "grid.linestyle"   : "--",
    "grid.alpha"       : 0.5,
    "legend.facecolor" : "#22263a",
    "legend.edgecolor" : "#2d3150",
    "font.size"        : 11,
    "axes.titlesize"   : 13,
    "axes.titleweight" : "bold",
})

ACCENT = "#6c63ff"
GREEN  = "#00d4aa"
RED    = "#ff6b6b"
YELLOW = "#ffd93d"
WHITE  = "#c9d1e9"

x        = df["n_workers"]
x_labels = x.astype(str)

cluster_info = ""
if "cluster_nodes" in df.columns:
    cluster_info = (f"Cluster: {int(df['cluster_nodes'].iloc[0])} nós, "
                    f"{int(df['cluster_total_cpus'].iloc[0])} CPUs")

# ─── GRÁFICO 1: SPEEDUP ──────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(9, 5))
fig.suptitle(f"Speedup Real vs. Ideal — Ray Data (QP3)\n{cluster_info}", fontsize=12)

ax.plot(x, df["speedup_real"],
        marker="o", color=ACCENT, linewidth=2.5, markersize=9,
        label="Speedup real", zorder=3)
ax.plot(x, df["speedup_ideal"],
        marker="^", color=YELLOW, linewidth=1.8, markersize=8,
        linestyle="--", label="Speedup ideal (linear)", zorder=3)
ax.fill_between(x, df["speedup_real"], df["speedup_ideal"],
                color=RED, alpha=0.13, label="Gap (overhead)")

# Anota os valores de speedup real em cada ponto
for _, row in df.iterrows():
    ax.annotate(f"{row['speedup_real']:.2f}×",
                xy=(row["n_workers"], row["speedup_real"]),
                xytext=(0, 10), textcoords="offset points",
                ha="center", color=WHITE, fontsize=9)

ax.set_xlabel("Número de Workers (concurrency)")
ax.set_ylabel("Speedup (×)")
ax.set_xticks(x)
ax.legend()
ax.grid(zorder=0)
plt.tight_layout()
plt.savefig("grafico_speedup.png", dpi=150, bbox_inches="tight")
print("✓ grafico_speedup.png")
plt.close()

# ─── GRÁFICO 2: EFICIÊNCIA ───────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(9, 5))
fig.suptitle(f"Eficiência Paralela (%) — Ray Data (QP3)\n{cluster_info}", fontsize=12)

colors = [GREEN if e >= 70 else YELLOW if e >= 50 else RED
          for e in df["efficiency_pct"]]

bars = ax.bar(x_labels, df["efficiency_pct"], color=colors, alpha=0.88, zorder=3, width=0.5)

# Valor dentro de cada barra
for bar, val in zip(bars, df["efficiency_pct"]):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() - 4,
            f"{val:.1f}%",
            ha="center", va="top", color="#0f1117", fontsize=10, fontweight="bold")

ax.axhline(100, color=YELLOW, linestyle="--", linewidth=1.2, label="100% (linear ideal)")
ax.axhline(70,  color=GREEN,  linestyle=":",  linewidth=1.2, label="70% (limiar aceitável)")
ax.set_xlabel("Número de Workers (concurrency)")
ax.set_ylabel("Eficiência (%)")
ax.set_ylim(0, 120)
ax.legend(fontsize=9)
ax.grid(axis="y", zorder=0)
plt.tight_layout()
plt.savefig("grafico_eficiencia.png", dpi=150, bbox_inches="tight")
print("✓ grafico_eficiencia.png")
plt.close()

# ─── GRÁFICO 3: TEMPO ────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(9, 5))
fig.suptitle(f"Tempo Médio de Execução — Ray Data (QP3)\n{cluster_info}", fontsize=12)

ax.bar(x_labels, df["mean_time_s"], color=ACCENT, alpha=0.85, zorder=3, width=0.5)
ax.errorbar(x_labels, df["mean_time_s"], yerr=df["std_time_s"],
            fmt="none", color=WHITE, capsize=6, linewidth=1.8, zorder=4)

for i, row in df.iterrows():
    ax.text(i, row["mean_time_s"] + row["std_time_s"] + 0.5,
            f"{row['mean_time_s']:.1f}s",
            ha="center", color=WHITE, fontsize=9)

ax.set_xlabel("Número de Workers (concurrency)")
ax.set_ylabel("Tempo médio (s)")
ax.grid(axis="y", zorder=0)
plt.tight_layout()
plt.savefig("grafico_tempo.png", dpi=150, bbox_inches="tight")
print("✓ grafico_tempo.png")
plt.close()

# ─── RESUMO FINAL ─────────────────────────────────────────────────────────────

print("\n" + "=" * 65)
print("  RESUMO — EXP 3 (QP3)")
print(f"  {'Workers':>8} {'Tempo (s)':>10} {'Std':>6} "
      f"{'emb/s':>8} {'Speedup':>9} {'Efic%':>8}")
print("-" * 65)
for _, row in df.iterrows():
    print(f"  {int(row['n_workers']):>8} "
          f"{row['mean_time_s']:>10.2f} "
          f"{row['std_time_s']:>6.2f} "
          f"{row['mean_throughput_emb_s']:>8.1f} "
          f"{row['speedup_real']:>8.2f}x "
          f"{row['efficiency_pct']:>8.1f}%")

print(f"\n✓ Gráficos gerados:")
for f in ["grafico_speedup.png", "grafico_eficiencia.png", "grafico_tempo.png"]:
    print(f"  {f}")
print("\nPara baixar: scp ubuntu@<HEAD_IP>:~/ray-exp/*.png ./resultados/")